In [2]:
from bs4 import BeautifulSoup
import requests
import pandas as pd

In [3]:
# download site, run through each item and scrap the data into table
def scrap_products_from_page(page):
    # scrapped site
    allo_site = BeautifulSoup(page.text, 'html')
    
    # find product layout
    products_layout = allo_site.find('div', class_ = 'products-layout__container products-layout--grid products-layout--is-tab-list')
    
    # find all product in a product layout
    item_layouts = products_layout.find_all('div', class_ = 'products-layout__item')
    
    for item in item_layouts:
        # empty list that will contain all the info about one product, each entry for one column value
        product_info = []

        # fill product id product id
        product_id = item.find(class_ ='product-sku__value').text.strip()
        product_info.append(product_id)
    
        # find title information
        title_info = item.find('a', class_ = 'product-card__title').text.strip()
    
        # fill company column - first word in a title
        company = title_info.split(' ')[0]
        product_info.append(company)
    
        # fill product name - all words in title except first
        product_name = ' '.join(title_info.split(' ')[1:])
        product_info.append(product_name)
    
        # fill price info
        try:
            price_info = item.find(class_ = 'sum').text.strip()
        except AttributeError:
            price_info = 0
        product_info.append(price_info)
    
        # fill rating info leave numeric part, if it exists, else fill 0
        div_star_tag = item.find('div', class_ = 'rating-stars__secondary')
        try:
            style_content = div_star_tag.get("style")
            rating_info = style_content.split(':')[1]
        except AttributeError:
             rating_info = 0
        product_info.append(rating_info)
    
        # fill review info, if it exists, else fill 0
        review_tag = item.find(class_ = 'review-button__text review-button__text--count')
        try:
            review_info = review_tag.text.strip()
        except AttributeError:
            review_info = 0
        product_info.append(review_info)
    
        # writing down data into table
        length = len(df)
        df.loc[length] = product_info

In [8]:
# creating table with columns listed 
df = pd.DataFrame(columns = ['Product ID','Company','Product Name', 'Price', 'Rating', 'Reviews'])

for page_num in range(1,51):
    # url for given site
    url =  'https://allo.ua/ua/products/mobile/p-' + str(page_num)

    # ask get sites code
    page = requests.get(url)
    
    scrap_products_from_page(page)
df

,Product ID,Company,Product Name,Price,Rating,Reviews
0,1213375,Samsung,Galaxy Fold 8 Ultra 16/1TB Graphite (SM-F976BZ...,117 999,100%;,2
1,1213385,Samsung,Galaxy Fold 8 12/512GB Graphite (SM-F971BZKCSEK),90 999,100%;,2
2,1213393,Samsung,Galaxy Flip 8 12/512GB Pink (SM-F776BLIHSEK),64 999,0,0
3,1183684,Xiaomi,REDMI Note 15 Pro 8/256GB Black,14 999,96.42399999999999%;,194
4,1170490,Apple,iPhone 17 Pro 256GB Cosmic Orange (MG8H4),66 999,98.908%;,805
...,...,...,...,...,...,...
2995,29631662-3025,Смартфон,iPhone 14 Pro 256 e-Sim USA Gold Seller Refurb...,39 000,0,0
2996,29172610-1228,Мобільний,телефон Sigma X-style 171 MINI Track Black-Ora...,787,0,0
2997,28885419-0789,Мобільний,телефон Nomi i1880 Red (640898-01),589,0,0
2998,29211206-5731,Смартфон,Oppo A6 Pro 8/256GB Stellar Blue (OFCPH2799 _B...,16 874,0%;,1


In [9]:
df.loc[df['Product ID'] == '']

,Product ID,Company,Product Name,Price,Rating,Reviews


In [10]:
df.to_csv(r'C:\Users\User\Downloads\Course Materials\Mobiles_scrapped12.csv')